# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'Unknown')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will inspect the record sets, fields, and columns. All entities are referenced by their `@id` (unique identifier in a Croissant schema).

In [ ]:
# List all record sets and their fields/columns by `@id`

record_sets = dataset.metadata.recordSets
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"\nRecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")
    fields = getattr(rs, 'fields', [])
    if fields:
        print("  Fields:")
        for field in fields:
            print(f"    Field @id: {field.id} (name: {field.name}, dataType: {field.dataType})")
    # For tabular record sets, list columns
    columns = getattr(rs, 'columns', [])
    if columns:
        print("  Columns:")
        for column in columns:
            print(f"    Column @id: {column.id} (name: {column.name}, dataType: {column.dataType})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract the main protein table. Below, the record set with the main data (typically named 'protein_table' or similar in these datasets) and its fields are referenced by `@id`. Please consult the previous cell output to pick an appropriate `@id`.

In [ ]:
# Let's select all tabular record sets (those with columns and records)

table_record_sets = []
for rs in dataset.metadata.recordSets:
    if getattr(rs, 'columns', None):
        table_record_sets.append(rs.id)

print(f"Tabular record sets: {table_record_sets}")

dataframes = {}
for rs_id in table_record_sets:
    print(f"Loading records from record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if len(records):
        df = pd.DataFrame(records)
        print(f"Columns in {rs_id}: {df.columns.tolist()}")
        print(df.head())
        dataframes[rs_id] = df
    else:
        print(f"No records extracted from {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates removing outliers, transforming data distributions, and grouping records using referenced `@id`s.

Pick a numeric field for analysis (for example, a column such as `cr:abundance` or `cr:MW`).

Replace the placeholder with a valid `@id` for the numeric field and a categorical/grouping field from your data (see column names from the previous cell's output).

In [ ]:
# Choose which tabular record set and fields to analyze
# Use the output above. For illustration, we'll use the first available record set and pick numeric & group fields by their @id or name.

if table_record_sets:
    record_set_id = table_record_sets[0]
    df = dataframes[record_set_id]
    print(f"Analyzing record set: {record_set_id}")

    # Attempt to pick common numeric and group columns
    # These may need to be adapted for your dataset fields
    numeric_candidates = [col for col in df.columns if any(word in col.lower() for word in ['abundance', 'count', 'weight', 'mw', 'intensity'])]
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Chosen numeric field: {numeric_field}")
    else:
        numeric_field = df.select_dtypes(include=['number']).columns[0]
        print(f"Chosen numeric field (by data type): {numeric_field}")

    group_candidates = [col for col in df.columns if any(word in col.lower() for word in ['group', 'condition', 'id', 'type', 'class', 'sample']) and col != numeric_field]
    group_field = group_candidates[0] if group_candidates else df.columns[0]
    print(f"Chosen group field: {group_field}")

    # Filtering
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]

    print(f"Filtered records with {numeric_field} > {threshold:.2f}: (showing up to 5)")
    print(filtered_df[[group_field, numeric_field]].head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print("Grouped data (mean) by group field:")
        print(grouped_df.head())
else:
    print("No tabular record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if table_record_sets and numeric_field and group_field:
    plt.figure(figsize=(8,6))
    filtered_df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Histogram of {numeric_field}')
    plt.show()

    # Boxplot by group
    if filtered_df[group_field].nunique() < 20:
        plt.figure(figsize=(10,6))
        filtered_df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric/group field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded using the `mlcroissant` library using its Croissant schema.
- Record sets and their field/column names and `@id`s were retrieved, providing insights into the schema.
- Tabular data was extracted for exploratory data analysis and visualizations.
- You can now use `mlcroissant` to further automate or validate data workflows for FAIR record-keeping and reproducibility.